# vLLM Baseline Benchmark

## Objective

The goal of this notebook is to establish a reproducible performance baseline for vLLM before modifying the scheduler policy.

The baseline will evaluate representative LLM inference workloads under controlled configurations and measure both latency and throughput behavior.

The same workloads and metrics will later be reused to compare scheduler optimizations.

## Benchmark Scope

The benchmark will cover several workload patterns:

- short prompt + short output
- long prompt + short output
- short prompt + long output
- mixed workloads with different prompt lengths

These workloads are designed to expose different inference bottlenecks:

- short prompts emphasize scheduling and launch overhead
- long prompts emphasize prefill cost
- long outputs emphasize decode performance
- mixed workloads expose scheduling interactions between prefill and decode

## Metrics

The benchmark will record:

- prompt token count
- output token count
- end-to-end latency
- output token throughput
- TTFT (Time To First Token)
- TPOT (Time Per Output Token)
- p50 latency
- p95 latency

Offline generation will first be used to validate the workload definitions and measure basic latency and throughput.

A streaming serving benchmark will then be used to measure TTFT and TPOT.

## Scheduler Configurations

Two scheduler configurations will be evaluated.

### Default Configuration

The default vLLM scheduler configuration will be used as the reference baseline.

Typical configuration:

- `max_num_batched_tokens = 8192`
- `enable_chunked_prefill = True`

### Controlled Token-Budget Configuration

A smaller scheduler token budget will also be evaluated:

- `max_num_batched_tokens = 512`
- `enable_chunked_prefill = True`

This configuration makes chunked prefill behavior easier to observe and provides a controlled environment for later scheduler experiments.

## Experimental Principle

The baseline stage does not modify the scheduler policy.

The purpose is to measure the behavior of the existing system under reproducible workloads.

The workflow is:

```text
Define workload
      ↓
Run baseline
      ↓
Collect latency and throughput metrics
      ↓
Analyze scheduler behavior
      ↓
Modify scheduler policy
      ↓
Repeat the same benchmark
      ↓
Compare against baseline

## 0. Confirm the GPU

In [1]:
!nvidia-smi

Mon Sep 21 01:13:50 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip uninstall -y torchaudio
!pip install -U torchaudio==2.11.0+cu130 \
  --index-url https://download.pytorch.org/whl/cu130

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu130
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 36.9 MB/s eta 0:00:00


In [3]:
%pip install -q -U vllm openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
# =============================================================================
# Common benchmark configuration
# =============================================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.70
DTYPE = "float16"

print("Model:", MODEL_NAME)
print("Max model length:", MAX_MODEL_LEN)
print("GPU memory utilization:", GPU_MEMORY_UTILIZATION)
print("Dtype:", DTYPE)

Model: Qwen/Qwen2.5-1.5B-Instruct
Max model length: 4096
GPU memory utilization: 0.7
Dtype: float16


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
CUDA version: 13.0
Device count: 1
GPU: Tesla T4


In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
Device count: 1
GPU: Tesla T4


## 1. Environment + workload definitions

## Environment and Workload Definitions

This section prepares the benchmark environment and defines a set of representative inference workloads.

The goal is to make later benchmark results reproducible and comparable across scheduler configurations.

The workloads cover four common inference patterns:

- short prompt + short output
- long prompt + short output
- short prompt + long output
- mixed prompt lengths

These workloads are designed to stress different parts of the inference pipeline, including scheduling overhead, prefill computation, decode performance, and mixed prefill/decode interactions.

In [4]:
# =============================================================================
# Workload definitions
# =============================================================================

workloads = {
    "short_short": {
        "description": "Short prompt with short output.",
        "prompts": [
            "What is a GPU warp?",
            "What is KV cache?",
            "What is tensor parallelism?",
            "What is FlashAttention?",
        ],
        "max_tokens": 32,
    },

    "long_short": {
        "description": "Long prompt with short output; primarily stresses prefill.",
        "prompts": [
            "Explain GPU memory hierarchy in detail. " * 300,
            "Explain transformer inference optimization in detail. " * 300,
            "Explain CUDA kernel optimization techniques in detail. " * 300,
            "Explain distributed LLM inference in detail. " * 300,
        ],
        "max_tokens": 32,
    },

    "short_long": {
        "description": "Short prompt with long output; primarily stresses decode.",
        "prompts": [
            "Explain how vLLM works.",
            "Explain CUDA memory optimization.",
            "Explain how FlashAttention works.",
            "Explain tensor parallelism for LLM inference.",
        ],
        "max_tokens": 256,
    },

    "mixed": {
        "description": "Mixed prompt lengths to expose scheduler interactions.",
        "prompts": [
            "Explain GPU memory hierarchy in detail. " * 300,
            "What is a GPU warp?",
            "Explain transformer inference optimization in detail. " * 200,
            "What is KV cache?",
        ],
        "max_tokens": 64,
    },
}

In [5]:
# =============================================================================
# Inspect workload definitions
# =============================================================================

for name, config in workloads.items():
    print("=" * 80)
    print("Workload:", name)
    print("Description:", config["description"])
    print("Number of prompts:", len(config["prompts"]))
    print("Max output tokens:", config["max_tokens"])

    for i, prompt in enumerate(config["prompts"]):
        print(
            f"  Prompt {i}: "
            f"{len(prompt)} characters"
        )

Workload: short_short
Description: Short prompt with short output.
Number of prompts: 4
Max output tokens: 32
  Prompt 0: 19 characters
  Prompt 1: 17 characters
  Prompt 2: 27 characters
  Prompt 3: 23 characters
Workload: long_short
Description: Long prompt with short output; primarily stresses prefill.
Number of prompts: 4
Max output tokens: 32
  Prompt 0: 12000 characters
  Prompt 1: 16200 characters
  Prompt 2: 16500 characters
  Prompt 3: 13500 characters
Workload: short_long
Description: Short prompt with long output; primarily stresses decode.
Number of prompts: 4
Max output tokens: 256
  Prompt 0: 23 characters
  Prompt 1: 33 characters
  Prompt 2: 33 characters
  Prompt 3: 45 characters
Workload: mixed
Description: Mixed prompt lengths to expose scheduler interactions.
Number of prompts: 4
Max output tokens: 64
  Prompt 0: 12000 characters
  Prompt 1: 19 characters
  Prompt 2: 10800 characters
  Prompt 3: 17 characters


### Token Count Validation

Character length is only a rough proxy for inference workload size.

For reproducible benchmarking, prompt length should be measured in model tokens because prefill cost, KV-cache usage, and scheduler token budgets are all token-based.

This section uses the model tokenizer to measure the actual prompt lengths for each workload.

In [16]:
# =============================================================================
# Load tokenizer
# =============================================================================

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

print("Tokenizer loaded:", MODEL_NAME)

Tokenizer loaded: Qwen/Qwen2.5-1.5B-Instruct


In [7]:
# =============================================================================
# Measure prompt token counts
# =============================================================================

workload_token_counts = {}

for name, config in workloads.items():
    token_counts = []

    print("=" * 80)
    print("Workload:", name)

    for i, prompt in enumerate(config["prompts"]):
        token_ids = tokenizer.encode(
            prompt,
            add_special_tokens=False,
        )

        token_count = len(token_ids)
        token_counts.append(token_count)

        print(
            f"  Prompt {i}: "
            f"{token_count:4d} tokens | "
            f"{len(prompt):5d} characters"
        )

    workload_token_counts[name] = token_counts

    print(
        "  Token range:",
        f"{min(token_counts)} - {max(token_counts)}"
    )

Workload: short_short
  Prompt 0:    6 tokens |    19 characters
  Prompt 1:    5 tokens |    17 characters
  Prompt 2:    6 tokens |    27 characters
  Prompt 3:    5 tokens |    23 characters
  Token range: 5 - 6
Workload: long_short
  Prompt 0: 2102 tokens | 12000 characters
  Prompt 1: 2102 tokens | 16200 characters
  Prompt 2: 2402 tokens | 16500 characters
  Prompt 3: 2402 tokens | 13500 characters
  Token range: 2102 - 2402
Workload: short_long
  Prompt 0:    8 tokens |    23 characters
  Prompt 1:    6 tokens |    33 characters
  Prompt 2:    7 tokens |    33 characters
  Prompt 3:   10 tokens |    45 characters
  Token range: 6 - 10
Workload: mixed
  Prompt 0: 2102 tokens | 12000 characters
  Prompt 1:    6 tokens |    19 characters
  Prompt 2: 1402 tokens | 10800 characters
  Prompt 3:    5 tokens |    17 characters
  Token range: 5 - 2102


In [8]:
# =============================================================================
# Workload token summary
# =============================================================================

for name, counts in workload_token_counts.items():
    print(
        f"{name:12s} | "
        f"min={min(counts):4d} | "
        f"max={max(counts):4d} | "
        f"mean={sum(counts) / len(counts):7.1f} | "
        f"max_output={workloads[name]['max_tokens']}"
    )

short_short  | min=   5 | max=   6 | mean=    5.5 | max_output=32
long_short   | min=2102 | max=2402 | mean= 2252.0 | max_output=32
short_long   | min=   6 | max=  10 | mean=    7.8 | max_output=256
mixed        | min=   5 | max=2102 | mean=  878.8 | max_output=64


In [9]:
# =============================================================================
# Build workload summary table
# =============================================================================

import pandas as pd

summary_rows = []

for name, counts in workload_token_counts.items():
    summary_rows.append({
        "workload": name,
        "num_requests": len(counts),
        "min_prompt_tokens": min(counts),
        "max_prompt_tokens": max(counts),
        "mean_prompt_tokens": sum(counts) / len(counts),
        "max_output_tokens": workloads[name]["max_tokens"],
    })

workload_summary = pd.DataFrame(summary_rows)

workload_summary

,workload,num_requests,min_prompt_tokens,max_prompt_tokens,mean_prompt_tokens,max_output_tokens
0,short_short,4,5,6,5.50,32
1,long_short,4,2102,2402,2252.00,32
2,short_long,4,6,10,7.75,256
3,mixed,4,5,2102,878.75,64


## 2. Offline Baseline Benchmark

This section measures basic end-to-end inference performance using the offline `LLM.generate()` API.

The goal is to establish a simple and reproducible baseline before moving to streaming serving benchmarks.

The offline benchmark records:

- prompt token count
- output token count
- end-to-end latency
- total generated tokens
- output-token throughput

This stage does not measure TTFT or TPOT because `LLM.generate()` is a blocking API and does not expose token arrival times.

In [10]:
# =============================================================================
# Offline benchmark helper
# =============================================================================

import time
import statistics
from vllm import LLM, SamplingParams


def run_offline_benchmark(
    llm,
    workload_name,
    workload_config,
    num_repeats=3,
):
    prompts = workload_config["prompts"]

    sampling_params = SamplingParams(
        temperature=0.0,
        max_tokens=workload_config["max_tokens"],
    )

    runs = []

    for repeat in range(num_repeats):
        start_time = time.perf_counter()

        outputs = llm.generate(
            prompts,
            sampling_params,
            use_tqdm=False,
        )

        end_time = time.perf_counter()

        latency = end_time - start_time

        prompt_tokens = sum(
            len(output.prompt_token_ids)
            for output in outputs
        )

        output_tokens = sum(
            len(output.outputs[0].token_ids)
            for output in outputs
        )

        output_throughput = (
            output_tokens / latency
            if latency > 0
            else 0.0
        )

        runs.append({
            "workload": workload_name,
            "repeat": repeat + 1,
            "num_requests": len(outputs),
            "prompt_tokens": prompt_tokens,
            "output_tokens": output_tokens,
            "latency_s": latency,
            "output_tokens_per_s": output_throughput,
        })

    return runs

In [11]:
# =============================================================================
# Default scheduler baseline
# =============================================================================
from vllm import LLM, SamplingParams

llm_default = LLM(
    model=MODEL_NAME,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    dtype=DTYPE,
)

INFO 09-21 01:20:45 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
INFO 09-21 01:21:05 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-21 01:21:05 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-21 01:21:05 [model.py:2021] Using max model len 4096
INFO 09-21 01:21:05 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-21 01:21:06 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

WARNING 09-21 01:21:10 [system_utils.py:157] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 09-21 01:24:13 [hf.py:547] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


In [12]:
# =============================================================================
# Smoke test
# =============================================================================

smoke_result = run_offline_benchmark(
    llm=llm_default,
    workload_name="short_short",
    workload_config=workloads["short_short"],
    num_repeats=1,
)

smoke_result

[{'workload': 'short_short',
  'repeat': 1,
  'num_requests': 4,
  'prompt_tokens': 22,
  'output_tokens': 128,
  'latency_s': 0.6239701940000941,
  'output_tokens_per_s': 205.1380037553215}]

In [13]:
# =============================================================================
# Run default offline baseline
# =============================================================================

default_results = []

for workload_name, workload_config in workloads.items():
    print("=" * 80)
    print("Running workload:", workload_name)

    results = run_offline_benchmark(
        llm=llm_default,
        workload_name=workload_name,
        workload_config=workload_config,
        num_repeats=3,
    )

    default_results.extend(results)

    for r in results:
        print(
            f"repeat={r['repeat']} | "
            f"latency={r['latency_s']:.4f}s | "
            f"output_tokens={r['output_tokens']} | "
            f"throughput={r['output_tokens_per_s']:.2f} tok/s"
        )

Running workload: short_short
repeat=1 | latency=0.5911s | output_tokens=128 | throughput=216.56 tok/s
repeat=2 | latency=0.6230s | output_tokens=128 | throughput=205.44 tok/s
repeat=3 | latency=0.6317s | output_tokens=128 | throughput=202.64 tok/s
Running workload: long_short
repeat=1 | latency=10.0789s | output_tokens=128 | throughput=12.70 tok/s
repeat=2 | latency=0.7727s | output_tokens=128 | throughput=165.66 tok/s
repeat=3 | latency=0.8130s | output_tokens=128 | throughput=157.44 tok/s
Running workload: short_long
repeat=1 | latency=3.9666s | output_tokens=916 | throughput=230.93 tok/s
repeat=2 | latency=3.9555s | output_tokens=916 | throughput=231.58 tok/s
repeat=3 | latency=3.9528s | output_tokens=916 | throughput=231.74 tok/s
Running workload: mixed
repeat=1 | latency=1.2757s | output_tokens=256 | throughput=200.67 tok/s
repeat=2 | latency=1.2759s | output_tokens=256 | throughput=200.64 tok/s
repeat=3 | latency=1.2851s | output_tokens=256 | throughput=199.21 tok/s


In [14]:
import pandas as pd

default_df = pd.DataFrame(default_results)

default_df

,workload,repeat,num_requests,prompt_tokens,output_tokens,latency_s,output_tokens_per_s
0,short_short,1,4,22,128,0.591073,216.555306
1,short_short,2,4,22,128,0.623049,205.441253
2,short_short,3,4,22,128,0.631655,202.642376
3,long_short,1,4,9008,128,10.078901,12.699797
4,long_short,2,4,9008,128,0.772678,165.657727
5,long_short,3,4,9008,128,0.813033,157.435100
6,short_long,1,4,31,916,3.966649,230.925378
7,short_long,2,4,31,916,3.955517,231.575276
8,short_long,3,4,31,916,3.952762,231.736697
9,mixed,1,4,3515,256,1.275712,200.672207


In [15]:
# =============================================================================
# Aggregate default baseline results
# =============================================================================

default_summary = (
    default_df
    .groupby("workload")
    .agg(
        mean_latency_s=("latency_s", "mean"),
        std_latency_s=("latency_s", "std"),
        mean_output_tokens_per_s=("output_tokens_per_s", "mean"),
        min_latency_s=("latency_s", "min"),
        max_latency_s=("latency_s", "max"),
    )
    .reset_index()
)

default_summary

,workload,mean_latency_s,std_latency_s,mean_output_tokens_per_s,min_latency_s,max_latency_s
0,long_short,3.888204,5.361339,111.930875,0.772678,10.078901
1,mixed,1.278915,0.005353,200.172072,1.275712,1.285094
2,short_long,3.958310,0.007353,231.412450,3.952762,3.966649
3,short_short,0.615259,0.021383,208.212978,0.591073,0.631655


In [16]:
# =============================================================================
# Inspect individual baseline runs
# =============================================================================

default_df[
    [
        "workload",
        "repeat",
        "prompt_tokens",
        "output_tokens",
        "latency_s",
        "output_tokens_per_s",
    ]
]

,workload,repeat,prompt_tokens,output_tokens,latency_s,output_tokens_per_s
0,short_short,1,22,128,0.591073,216.555306
1,short_short,2,22,128,0.623049,205.441253
2,short_short,3,22,128,0.631655,202.642376
3,long_short,1,9008,128,10.078901,12.699797
4,long_short,2,9008,128,0.772678,165.657727
5,long_short,3,9008,128,0.813033,157.435100
6,short_long,1,31,916,3.966649,230.925378
7,short_long,2,31,916,3.955517,231.575276
8,short_long,3,31,916,3.952762,231.736697
9,mixed,1,3515,256,1.275712,200.672207


In [17]:
def run_offline_benchmark(
    llm,
    workload_name,
    workload_config,
    num_repeats=3,
    warmup_runs=1,
):
    prompts = workload_config["prompts"]

    sampling_params = SamplingParams(
        temperature=0.0,
        max_tokens=workload_config["max_tokens"],
    )

    # -------------------------------------------------------------------------
    # Warm-up
    # -------------------------------------------------------------------------
    for _ in range(warmup_runs):
        llm.generate(
            prompts,
            sampling_params,
            use_tqdm=False,
        )

    # -------------------------------------------------------------------------
    # Measured runs
    # -------------------------------------------------------------------------
    runs = []

    for repeat in range(num_repeats):
        start_time = time.perf_counter()

        outputs = llm.generate(
            prompts,
            sampling_params,
            use_tqdm=False,
        )

        end_time = time.perf_counter()

        latency = end_time - start_time

        prompt_tokens = sum(
            len(output.prompt_token_ids)
            for output in outputs
        )

        output_tokens = sum(
            len(output.outputs[0].token_ids)
            for output in outputs
        )

        output_throughput = (
            output_tokens / latency
            if latency > 0
            else 0.0
        )

        runs.append({
            "workload": workload_name,
            "repeat": repeat + 1,
            "num_requests": len(outputs),
            "prompt_tokens": prompt_tokens,
            "output_tokens": output_tokens,
            "latency_s": latency,
            "output_tokens_per_s": output_throughput,
        })

    return runs

In [18]:
default_results = []

for workload_name, workload_config in workloads.items():
    print("=" * 80)
    print("Running workload:", workload_name)

    results = run_offline_benchmark(
        llm=llm_default,
        workload_name=workload_name,
        workload_config=workload_config,
        num_repeats=3,
        warmup_runs=1,
    )

    default_results.extend(results)

    for r in results:
        print(
            f"repeat={r['repeat']} | "
            f"latency={r['latency_s']:.4f}s | "
            f"throughput={r['output_tokens_per_s']:.2f} tok/s"
        )

Running workload: short_short
repeat=1 | latency=0.5199s | throughput=246.19 tok/s
repeat=2 | latency=0.7039s | throughput=181.83 tok/s
repeat=3 | latency=0.5975s | throughput=214.22 tok/s
Running workload: long_short
repeat=1 | latency=0.9914s | throughput=129.11 tok/s
repeat=2 | latency=0.8295s | throughput=154.32 tok/s
repeat=3 | latency=0.8070s | throughput=158.60 tok/s
Running workload: short_long
repeat=1 | latency=4.0616s | throughput=225.52 tok/s
repeat=2 | latency=3.9831s | throughput=233.74 tok/s
repeat=3 | latency=3.9814s | throughput=230.07 tok/s
Running workload: mixed
repeat=1 | latency=1.2920s | throughput=198.14 tok/s
repeat=2 | latency=1.2870s | throughput=198.91 tok/s
repeat=3 | latency=1.2920s | throughput=198.14 tok/s


In [20]:
default_df = pd.DataFrame(default_results)

default_summary = (
    default_df
    .groupby("workload")
    .agg(
        mean_latency_s=("latency_s", "mean"),
        std_latency_s=("latency_s", "std"),
        mean_output_tokens_per_s=("output_tokens_per_s", "mean"),
        min_latency_s=("latency_s", "min"),
        max_latency_s=("latency_s", "max"),
    )
    .reset_index()
)

default_summary

,workload,mean_latency_s,std_latency_s,mean_output_tokens_per_s,min_latency_s,max_latency_s
0,long_short,0.875965,0.100585,147.344066,0.807041,0.991387
1,mixed,1.290331,0.002884,198.399341,1.287001,1.292005
2,short_long,4.008738,0.045829,229.775935,3.981424,4.061647
3,short_short,0.607124,0.092382,214.082221,0.519927,0.703939


### Controlled 512-token Budget Baseline

In [24]:
# =============================================================================
# Release the default vLLM engine
# =============================================================================

del llm_default

import gc
import torch
import time

gc.collect()
torch.cuda.empty_cache()

time.sleep(2)

print(
    "Allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GiB"
)

print(
    "Reserved:",
    round(torch.cuda.memory_reserved() / 1024**3, 2),
    "GiB"
)

INFO 09-21 01:34:29 [utils.py:620] [shutdown] Process manager: send sigterm to process EngineCore
WARNING 09-21 01:34:34 [utils.py:640] [shutdown] Process manager: force killing remaining processes count=1
WARNING 09-21 01:34:34 [utils.py:645] [shutdown] Process manager: force killing remaining process EngineCore pid 5039
Allocated: 0.0 GiB
Reserved: 0.0 GiB


In [25]:
!nvidia-smi

Mon Sep 21 01:34:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [26]:
# =============================================================================
# Controlled 512-token scheduler configuration
# =============================================================================

llm_512 = LLM(
    model=MODEL_NAME,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    dtype=DTYPE,
    max_num_batched_tokens=512,
    enable_chunked_prefill=True,
)

config_512 = llm_512.llm_engine.vllm_config.scheduler_config

print("max_num_batched_tokens:", config_512.max_num_batched_tokens)
print("max_num_scheduled_tokens:", config_512.max_num_scheduled_tokens)
print("enable_chunked_prefill:", config_512.enable_chunked_prefill)

INFO 09-21 01:34:46 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.7, 'max_num_batched_tokens': 512, 'disable_log_stats': True, 'enable_chunked_prefill': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
INFO 09-21 01:34:46 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-21 01:34:46 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-21 01:34:46 [model.py:2021] Using max model len 4096
INFO 09-21 01:34:46 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
max_num_batched_tokens: 512
max_num_scheduled_tokens: None
enable_chunked_prefill: True


In [27]:
# =============================================================================
# Run 512-token offline baseline
# =============================================================================

results_512 = []

for workload_name, workload_config in workloads.items():
    print("=" * 80)
    print("Running workload:", workload_name)

    results = run_offline_benchmark(
        llm=llm_512,
        workload_name=workload_name,
        workload_config=workload_config,
        num_repeats=3,
        warmup_runs=1,
    )

    results_512.extend(results)

    for r in results:
        print(
            f"repeat={r['repeat']} | "
            f"latency={r['latency_s']:.4f}s | "
            f"throughput={r['output_tokens_per_s']:.2f} tok/s"
        )

Running workload: short_short
repeat=1 | latency=0.5020s | throughput=255.00 tok/s
repeat=2 | latency=1.4376s | throughput=89.04 tok/s
repeat=3 | latency=0.5348s | throughput=239.32 tok/s
Running workload: long_short
repeat=1 | latency=0.7745s | throughput=165.27 tok/s
repeat=2 | latency=0.7741s | throughput=165.35 tok/s
repeat=3 | latency=0.7752s | throughput=165.11 tok/s
Running workload: short_long
repeat=1 | latency=3.9588s | throughput=231.38 tok/s
repeat=2 | latency=4.0017s | throughput=228.90 tok/s
repeat=3 | latency=3.9716s | throughput=230.64 tok/s
Running workload: mixed
repeat=1 | latency=1.2901s | throughput=198.43 tok/s
repeat=2 | latency=1.2998s | throughput=196.95 tok/s
repeat=3 | latency=1.3134s | throughput=194.92 tok/s


In [28]:
# =============================================================================
# Aggregate 512-token baseline
# =============================================================================

df_512 = pd.DataFrame(results_512)

summary_512 = (
    df_512
    .groupby("workload")
    .agg(
        mean_latency_s=("latency_s", "mean"),
        std_latency_s=("latency_s", "std"),
        mean_output_tokens_per_s=("output_tokens_per_s", "mean"),
        min_latency_s=("latency_s", "min"),
        max_latency_s=("latency_s", "max"),
    )
    .reset_index()
)

summary_512

,workload,mean_latency_s,std_latency_s,mean_output_tokens_per_s,min_latency_s,max_latency_s
0,long_short,0.774605,0.000571,165.245645,0.774092,0.775221
1,mixed,1.301108,0.011667,196.765879,1.290137,1.313365
2,short_long,3.977372,0.022030,230.307529,3.958804,4.001714
3,short_short,0.824804,0.530949,194.451482,0.501967,1.437598


In [29]:
# =============================================================================
# Compare default and 512-token baselines
# =============================================================================

comparison = default_summary.merge(
    summary_512,
    on="workload",
    suffixes=("_default", "_512"),
)

comparison["latency_change_pct"] = (
    (
        comparison["mean_latency_s_512"]
        - comparison["mean_latency_s_default"]
    )
    / comparison["mean_latency_s_default"]
    * 100
)

comparison["throughput_change_pct"] = (
    (
        comparison["mean_output_tokens_per_s_512"]
        - comparison["mean_output_tokens_per_s_default"]
    )
    / comparison["mean_output_tokens_per_s_default"]
    * 100
)

comparison[
    [
        "workload",
        "mean_latency_s_default",
        "mean_latency_s_512",
        "latency_change_pct",
        "mean_output_tokens_per_s_default",
        "mean_output_tokens_per_s_512",
        "throughput_change_pct",
    ]
]

,workload,mean_latency_s_default,mean_latency_s_512,latency_change_pct,mean_output_tokens_per_s_default,mean_output_tokens_per_s_512,throughput_change_pct
0,long_short,0.875965,0.774605,-11.571256,147.344066,165.245645,12.149508
1,mixed,1.290331,1.301108,0.835217,198.399341,196.765879,-0.823320
2,short_long,4.008738,3.977372,-0.782433,229.775935,230.307529,0.231353
3,short_short,0.607124,0.824804,35.854445,214.082221,194.451482,-9.169719


## 3. Default vs Controlled Scheduler Configuration

This section compares two scheduler configurations using the same offline workloads.

### Default configuration

- `max_num_batched_tokens = 8192`
- `enable_chunked_prefill = True`

### Controlled configuration

- `max_num_batched_tokens = 512`
- `enable_chunked_prefill = True`

The controlled configuration intentionally reduces the per-step token budget so that long prefills are split across multiple scheduler iterations.

The comparison focuses on:

- mean end-to-end latency
- output-token throughput
- latency change relative to the default configuration
- throughput change relative to the default configuration

A negative latency change indicates lower latency under the controlled configuration.

A positive throughput change indicates higher output-token throughput under the controlled configuration.

In [30]:
# =============================================================================
# Compare default and controlled scheduler configurations
# =============================================================================

comparison = default_summary.merge(
    summary_512,
    on="workload",
    suffixes=("_default", "_512"),
)

comparison["latency_change_pct"] = (
    (
        comparison["mean_latency_s_512"]
        - comparison["mean_latency_s_default"]
    )
    / comparison["mean_latency_s_default"]
    * 100
)

comparison["throughput_change_pct"] = (
    (
        comparison["mean_output_tokens_per_s_512"]
        - comparison["mean_output_tokens_per_s_default"]
    )
    / comparison["mean_output_tokens_per_s_default"]
    * 100
)

comparison_table = comparison[
    [
        "workload",
        "mean_latency_s_default",
        "mean_latency_s_512",
        "latency_change_pct",
        "mean_output_tokens_per_s_default",
        "mean_output_tokens_per_s_512",
        "throughput_change_pct",
    ]
].copy()

comparison_table

,workload,mean_latency_s_default,mean_latency_s_512,latency_change_pct,mean_output_tokens_per_s_default,mean_output_tokens_per_s_512,throughput_change_pct
0,long_short,0.875965,0.774605,-11.571256,147.344066,165.245645,12.149508
1,mixed,1.290331,1.301108,0.835217,198.399341,196.765879,-0.823320
2,short_long,4.008738,3.977372,-0.782433,229.775935,230.307529,0.231353
3,short_short,0.607124,0.824804,35.854445,214.082221,194.451482,-9.169719


In [31]:
# =============================================================================
# Format comparison table
# =============================================================================

comparison_table_rounded = comparison_table.copy()

numeric_columns = [
    "mean_latency_s_default",
    "mean_latency_s_512",
    "latency_change_pct",
    "mean_output_tokens_per_s_default",
    "mean_output_tokens_per_s_512",
    "throughput_change_pct",
]

comparison_table_rounded[numeric_columns] = (
    comparison_table_rounded[numeric_columns].round(2)
)

comparison_table_rounded

,workload,mean_latency_s_default,mean_latency_s_512,latency_change_pct,mean_output_tokens_per_s_default,mean_output_tokens_per_s_512,throughput_change_pct
0,long_short,0.88,0.77,-11.57,147.34,165.25,12.15
1,mixed,1.29,1.30,0.84,198.40,196.77,-0.82
2,short_long,4.01,3.98,-0.78,229.78,230.31,0.23
3,short_short,0.61,0.82,35.85,214.08,194.45,-9.17


### Initial Observations

The controlled 512-token scheduler budget produced different effects across workloads.

- **Long prompt + short output:** latency decreased and output-token throughput increased.
- **Mixed workload:** performance remained close to the default configuration.
- **Short prompt + long output:** performance was nearly unchanged, suggesting limited sensitivity to the prefill token budget for decode-heavy workloads.
- **Short prompt + short output:** the controlled result showed high run-to-run variance, so this result should not yet be treated as a stable performance trend.

These offline measurements are useful for identifying broad behavior, but they do not expose request-level TTFT or TPOT.

A streaming benchmark is required to determine whether smaller prefill chunks improve responsiveness for short requests when long prefills are present.

## 4. Streaming Benchmark

The offline benchmark measures total batch completion time, but it does not expose when each request receives its first generated token.

For scheduler analysis, request-level streaming metrics are more informative because they reveal how prefill and decode scheduling affect user-visible latency.

This section uses the vLLM OpenAI-compatible server with streaming responses to measure:

- TTFT (Time To First Token)
- TPOT (Time Per Output Token)
- end-to-end request latency
- output-token throughput
- p50 and p95 latency statistics

### Why Streaming Metrics Matter

A long prefill request may occupy a large portion of the scheduler token budget.

Even if total batch completion time remains similar, the scheduling policy can significantly affect when shorter requests begin generating.

For example:

```text
Long prefill request
        +
Short request arrives
        ↓
Scheduler decides how token budget is shared
        ↓
Short request receives its first token

### OpenAI-Compatible Server Setup

The streaming benchmark uses vLLM's OpenAI-compatible HTTP server.

Running inference through the server makes it possible to observe token arrival times from streamed responses, which is required for measuring TTFT and TPOT.

The server will first be launched with the default scheduler configuration and later repeated with the controlled 512-token configuration.

In [ ]:
import os
os.kill(os.getpid(), 9)

In [7]:
!nvidia-smi

Mon Sep 21 01:52:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P0             28W /   70W |     645MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# =============================================================================
# Common benchmark configuration
# =============================================================================

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

MAX_MODEL_LEN = 4096
GPU_MEMORY_UTILIZATION = 0.70
DTYPE = "float16"

print("Model:", MODEL_NAME)
print("Max model length:", MAX_MODEL_LEN)
print("GPU memory utilization:", GPU_MEMORY_UTILIZATION)
print("Dtype:", DTYPE)

Model: Qwen/Qwen2.5-1.5B-Instruct
Max model length: 4096
GPU memory utilization: 0.7
Dtype: float16


In [4]:
# =============================================================================
# Start vLLM OpenAI-compatible server
# =============================================================================

import subprocess
import time
import os

SERVER_LOG = "/content/vllm_server.log"

server_process = subprocess.Popen(
    [
        "python",
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--model",
        MODEL_NAME,
        "--dtype",
        DTYPE,
        "--max-model-len",
        str(MAX_MODEL_LEN),
        "--gpu-memory-utilization",
        str(GPU_MEMORY_UTILIZATION),
        "--max-num-batched-tokens",
        "8192",
        "--enable-chunked-prefill",
        "--port",
        "8000",
    ],
    stdout=open(SERVER_LOG, "w"),
    stderr=subprocess.STDOUT,
)

print("Server PID:", server_process.pid)
print("Log:", SERVER_LOG)

Server PID: 13374
Log: /content/vllm_server.log


In [8]:
# =============================================================================
# Inspect server startup
# =============================================================================

import time
from pathlib import Path

time.sleep(60)

log_path = Path(SERVER_LOG)

if log_path.exists():
    print(log_path.read_text()[-5000:])

ache-memory=6263014298` (5.83 GiB) to fit into requested memory, or `--kv-cache-memory=10844170752` (10.1 GiB) to fully utilize gpu memory. Current kv cache memory in use is 6.13 GiB.
(EngineCore pid=13523) INFO 09-21 01:52:42 [jit_monitor.py:85] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=13523) INFO 09-21 01:52:42 [core.py:361] init engine (profile, create kv cache, warmup model) took 21.95 s (compilation: 1.47 s)
(EngineCore pid=13523) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
(EngineCore pid=13523) INFO 09-21 01:52:44 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(APIServer pid=13374) INFO 09-21 01:52:45 [entry.py:135] Supported tasks: ['generate']
(APIServer pid=13374) Warning: You are sending unauthenticated requests to the HF 

In [9]:
# =============================================================================
# Check server health
# =============================================================================

import requests

response = requests.get(
    "http://127.0.0.1:8000/health",
    timeout=5,
)

print("Status code:", response.status_code)
print("Server ready:", response.status_code == 200)

Status code: 200
Server ready: True


In [10]:
from pathlib import Path

log_path = Path("/content/vllm_server.log")

print(log_path.read_text()[-8000:])

   | 27/51 [00:02<00:01, 15.24it/s]
Capturing CUDA graphs (PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 14.41it/s]
(EngineCore pid=13523) 
Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:01<00:00, 19.54it/s]
(EngineCore pid=13523) INFO 09-21 01:52:40 [model_runner.py:960] Graph capturing finished in 6 secs, took 0.15 GiB
(EngineCore pid=13523) INFO 09-21 01:52:40 [gpu_worker.py:797] CUDA graph pool memory: 0.15 GiB (actual), 0.34 GiB (estimated), difference: 0.19 GiB (130.3%).
(EngineCore pid=13523) INFO 09-21 01:52:40 [gpu_worker.py:860] Free memory on device (14.46/14.56 GiB) on startup. Desired GPU memory utilization is (0.7, 10.19 GiB). Actual usage is 3.24 GiB for consumed memory (weights + non-torch), 0.82 GiB for peak activation, and 0.15 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=6263014298` (5.83 GiB) to fit into requested memory, or `--kv-cache-memory=10844170752` (10.1 GiB) to fully utilize gpu memory. Current kv cach

In [11]:
print("Server process status:", server_process.poll())

Server process status: None


### Streaming Client

The client sends streaming completion requests to the vLLM OpenAI-compatible server and records token arrival timestamps.

For each request, the client measures:

- request start time
- first-token arrival time
- request completion time
- TTFT
- TPOT
- end-to-end latency
- output token count

This single-request test validates the timing logic before running concurrent mixed-arrival workloads.

In [12]:
# =============================================================================
# Streaming benchmark helper
# =============================================================================

import json
import time
import requests


API_URL = "http://127.0.0.1:8000/v1/completions"


def run_streaming_request(
    prompt,
    max_tokens=64,
    temperature=0.0,
):
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "stream": True,
    }

    start_time = time.perf_counter()
    first_token_time = None
    finish_time = None

    generated_text = ""
    stream_events = 0

    with requests.post(
        API_URL,
        json=payload,
        stream=True,
        timeout=300,
    ) as response:

        response.raise_for_status()

        for line in response.iter_lines():
            if not line:
                continue

            decoded = line.decode("utf-8")

            if not decoded.startswith("data: "):
                continue

            data = decoded[len("data: "):]

            if data == "[DONE]":
                finish_time = time.perf_counter()
                break

            event = json.loads(data)

            text = event["choices"][0].get("text", "")

            if text:
                now = time.perf_counter()

                if first_token_time is None:
                    first_token_time = now

                generated_text += text
                stream_events += 1

    if finish_time is None:
        finish_time = time.perf_counter()

    ttft = (
        first_token_time - start_time
        if first_token_time is not None
        else None
    )

    latency = finish_time - start_time

    return {
        "ttft_s": ttft,
        "latency_s": latency,
        "stream_events": stream_events,
        "generated_text": generated_text,
    }

In [13]:
# =============================================================================
# Streaming smoke test
# =============================================================================

result = run_streaming_request(
    prompt="What is a GPU warp?",
    max_tokens=32,
)

print("TTFT:", result["ttft_s"])
print("Latency:", result["latency_s"])
print("Stream events:", result["stream_events"])
print("Generated text:")
print(result["generated_text"])

TTFT: 0.1306652620000932
Latency: 0.6481324530000165
Stream events: 32
Generated text:
 I'm trying to understand the concept of a GPU warp, but I can't find any good explanation. Can someone provide an example or analogy that might help me


In [14]:
# =============================================================================
# Add output token count and TPOT
# =============================================================================

def add_streaming_metrics(result):
    output_token_ids = tokenizer.encode(
        result["generated_text"],
        add_special_tokens=False,
    )

    output_tokens = len(output_token_ids)

    if (
        result["ttft_s"] is not None
        and output_tokens > 1
    ):
        tpot = (
            result["latency_s"] - result["ttft_s"]
        ) / (output_tokens - 1)
    else:
        tpot = None

    result["output_tokens"] = output_tokens
    result["tpot_s"] = tpot
    result["output_tokens_per_s"] = (
        output_tokens / result["latency_s"]
        if result["latency_s"] > 0
        else None
    )

    return result

In [17]:
result = add_streaming_metrics(result)

print("TTFT:", result["ttft_s"])
print("TPOT:", result["tpot_s"])
print("Latency:", result["latency_s"])
print("Output tokens:", result["output_tokens"])
print("Output throughput:", result["output_tokens_per_s"])

TTFT: 0.1306652620000932
TPOT: 0.01669249003225559
Latency: 0.6481324530000165
Output tokens: 32
Output throughput: 49.372624147859455


### Mixed-Arrival Streaming Workload

This experiment evaluates scheduler responsiveness when a short request arrives shortly after a long request has already started.

The workload is designed to create contention between:

- long-request prefill
- short-request admission
- decode scheduling

The experiment records request-level TTFT, TPOT, and end-to-end latency.

The same workload will later be repeated under both the default 8192-token budget and the controlled 512-token budget.

In [18]:
# =============================================================================
# Concurrent streaming request helper
# =============================================================================

import asyncio
import aiohttp
import json
import time


API_URL = "http://127.0.0.1:8000/v1/completions"


async def run_streaming_request_async(
    session,
    request_name,
    prompt,
    max_tokens=64,
    temperature=0.0,
):
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "max_tokens": max_tokens,
        "temperature": temperature,
        "stream": True,
    }

    start_time = time.perf_counter()
    first_token_time = None
    finish_time = None

    generated_text = ""
    stream_events = 0

    async with session.post(
        API_URL,
        json=payload,
    ) as response:

        response.raise_for_status()

        buffer = b""

        async for chunk in response.content.iter_any():
            buffer += chunk

            while b"\n" in buffer:
                line, buffer = buffer.split(b"\n", 1)
                line = line.strip()

                if not line:
                    continue

                decoded = line.decode("utf-8")

                if not decoded.startswith("data: "):
                    continue

                data = decoded[len("data: "):]

                if data == "[DONE]":
                    finish_time = time.perf_counter()
                    break

                event = json.loads(data)

                text = event["choices"][0].get("text", "")

                if text:
                    now = time.perf_counter()

                    if first_token_time is None:
                        first_token_time = now

                    generated_text += text
                    stream_events += 1

            if finish_time is not None:
                break

    if finish_time is None:
        finish_time = time.perf_counter()

    output_token_ids = tokenizer.encode(
        generated_text,
        add_special_tokens=False,
    )

    output_tokens = len(output_token_ids)

    ttft = (
        first_token_time - start_time
        if first_token_time is not None
        else None
    )

    latency = finish_time - start_time

    if ttft is not None and output_tokens > 1:
        tpot = (
            latency - ttft
        ) / (output_tokens - 1)
    else:
        tpot = None

    return {
        "request": request_name,
        "ttft_s": ttft,
        "tpot_s": tpot,
        "latency_s": latency,
        "output_tokens": output_tokens,
        "stream_events": stream_events,
    }

In [19]:
# =============================================================================
# Mixed-arrival workload
# =============================================================================

LONG_PROMPT = (
    "Explain GPU memory hierarchy in detail. " * 300
)

SHORT_PROMPT = "What is a GPU warp?"

ARRIVAL_DELAY_S = 0.05

In [20]:
# =============================================================================
# Run mixed-arrival experiment
# =============================================================================

async def run_mixed_arrival_experiment():
    async with aiohttp.ClientSession() as session:

        long_task = asyncio.create_task(
            run_streaming_request_async(
                session=session,
                request_name="long",
                prompt=LONG_PROMPT,
                max_tokens=64,
            )
        )

        await asyncio.sleep(ARRIVAL_DELAY_S)

        short_task = asyncio.create_task(
            run_streaming_request_async(
                session=session,
                request_name="short",
                prompt=SHORT_PROMPT,
                max_tokens=32,
            )
        )

        long_result, short_result = await asyncio.gather(
            long_task,
            short_task,
        )

        return long_result, short_result

In [21]:
long_result, short_result = await run_mixed_arrival_experiment()

print("LONG REQUEST")
print(long_result)

print()

print("SHORT REQUEST")
print(short_result)

LONG REQUEST
{'request': 'long', 'ttft_s': 1.1188287710001532, 'tpot_s': 0.017719259920636794, 'latency_s': 2.235142146000271, 'output_tokens': 64, 'stream_events': 64}

SHORT REQUEST
{'request': 'short', 'ttft_s': 1.108701729999666, 'tpot_s': 0.018046676548399244, 'latency_s': 1.6681487030000426, 'output_tokens': 32, 'stream_events': 32}


In [22]:
# =============================================================================
# Repeat mixed-arrival experiment
# =============================================================================

import pandas as pd
import numpy as np

default_streaming_results = []

for repeat in range(5):
    long_result, short_result = await run_mixed_arrival_experiment()

    long_result["repeat"] = repeat + 1
    short_result["repeat"] = repeat + 1

    default_streaming_results.append(long_result)
    default_streaming_results.append(short_result)

default_streaming_df = pd.DataFrame(default_streaming_results)

default_streaming_df

,request,ttft_s,tpot_s,latency_s,output_tokens,stream_events,repeat
0,long,0.107082,0.018680,1.283935,64,64,1
1,short,0.112374,0.018728,0.692951,32,32,1
2,long,0.050856,0.018045,1.187713,64,64,2
3,short,0.066015,0.018156,0.628853,32,32,2
4,long,0.052898,0.018052,1.190185,64,64,3
5,short,0.066019,0.018202,0.630275,32,32,3
6,long,0.167449,0.017641,1.278843,64,64,4
7,short,0.124206,0.018181,0.687805,32,32,4
8,long,0.064275,0.018164,1.208586,64,64,5
9,short,0.073931,0.018341,0.642491,32,32,5


In [23]:
# =============================================================================
# Aggregate streaming metrics
# =============================================================================

default_streaming_summary = (
    default_streaming_df
    .groupby("request")
    .agg(
        mean_ttft_s=("ttft_s", "mean"),
        p50_ttft_s=("ttft_s", lambda x: np.percentile(x, 50)),
        p95_ttft_s=("ttft_s", lambda x: np.percentile(x, 95)),
        mean_tpot_s=("tpot_s", "mean"),
        p50_tpot_s=("tpot_s", lambda x: np.percentile(x, 50)),
        p95_tpot_s=("tpot_s", lambda x: np.percentile(x, 95)),
        mean_latency_s=("latency_s", "mean"),
        p95_latency_s=("latency_s", lambda x: np.percentile(x, 95)),
    )
    .reset_index()
)

default_streaming_summary

,request,mean_ttft_s,p50_ttft_s,p95_ttft_s,mean_tpot_s,p50_tpot_s,p95_tpot_s,mean_latency_s,p95_latency_s
0,long,0.088512,0.064275,0.155375,0.018117,0.018052,0.018577,1.229852,1.282916
1,short,0.088509,0.073931,0.121840,0.018321,0.018202,0.018651,0.656475,0.691922


### Prefix-Caching Control

Automatic prefix caching is disabled during scheduler-policy benchmarks.

Repeated benchmark prompts can otherwise reuse previously computed KV-cache blocks, substantially reducing prefill time and TTFT.

Because this experiment is intended to isolate the effect of scheduler token budget and chunked prefill, prefix caching is disabled for both the default and controlled configurations.

In [24]:
server_process.terminate()
server_process.wait()

print("Server stopped.")

Server stopped.


In [35]:
!nvidia-smi

Mon Sep 21 02:04:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             35W /   70W |   14129MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [32]:
import subprocess

SERVER_LOG = "/content/vllm_server.log"

server_process = subprocess.Popen(
    [
        "python",
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--model",
        MODEL_NAME,
        "--dtype",
        DTYPE,
        "--max-model-len",
        str(MAX_MODEL_LEN),
        "--gpu-memory-utilization",
        str(GPU_MEMORY_UTILIZATION),
        "--max-num-batched-tokens",
        "8192",
        "--enable-chunked-prefill",
        "--no-enable-prefix-caching",
        "--port",
        "8000",
    ],
    stdout=open(SERVER_LOG, "w"),
    stderr=subprocess.STDOUT,
)

print("Server PID:", server_process.pid)

Server PID: 16550


In [34]:
# =============================================================================
# Inspect server startup
# =============================================================================

import time
from pathlib import Path

time.sleep(60)

log_path = Path(SERVER_LOG)

if log_path.exists():
    print(log_path.read_text()[-5000:])

UnicodeDecodeError: 'utf-8' codec can't decode byte 0x88 in position 15644: invalid start byte

In [36]:
default_streaming_results = []

for repeat in range(5):
    long_result, short_result = await run_mixed_arrival_experiment()

    long_result["repeat"] = repeat + 1
    short_result["repeat"] = repeat + 1

    default_streaming_results.append(long_result)
    default_streaming_results.append(short_result)

default_streaming_df = pd.DataFrame(default_streaming_results)

default_streaming_df

,request,ttft_s,tpot_s,latency_s,output_tokens,stream_events,repeat
0,long,2.572197,0.020693,3.875858,64,64,1
1,short,2.521563,0.019161,3.115548,32,32,1
2,long,1.077432,0.022062,2.467310,64,64,2
3,short,1.057202,0.019405,1.658769,32,32,2
4,long,1.065425,0.022692,2.495030,64,64,3
5,short,1.067964,0.019430,1.670294,32,32,3
6,long,1.119221,0.022675,2.547775,64,64,4
7,short,1.102751,0.019616,1.710841,32,32,4
8,long,1.148836,0.023465,2.627120,64,64,5
9,short,1.150210,0.019932,1.768114,32,32,5


In [37]:
steady_default_df = default_streaming_df[
    default_streaming_df["repeat"] > 1
].copy()

steady_default_summary = (
    steady_default_df
    .groupby("request")
    .agg(
        mean_ttft_s=("ttft_s", "mean"),
        p50_ttft_s=("ttft_s", lambda x: np.percentile(x, 50)),
        p95_ttft_s=("ttft_s", lambda x: np.percentile(x, 95)),
        mean_tpot_s=("tpot_s", "mean"),
        p50_tpot_s=("tpot_s", lambda x: np.percentile(x, 50)),
        p95_tpot_s=("tpot_s", lambda x: np.percentile(x, 95)),
        mean_latency_s=("latency_s", "mean"),
        p95_latency_s=("latency_s", lambda x: np.percentile(x, 95)),
    )
    .reset_index()
)

steady_default_summary

,request,mean_ttft_s,p50_ttft_s,p95_ttft_s,mean_tpot_s,p50_tpot_s,p95_tpot_s,mean_latency_s,p95_latency_s
0,long,1.102729,1.098327,1.144393,0.022723,0.022684,0.023349,2.534309,2.615218
1,short,1.094532,1.085358,1.143091,0.019596,0.019523,0.019885,1.702005,1.759523


In [41]:
# =============================================================================
# Stop current server
# =============================================================================

server_process.terminate()
server_process.wait()

print("Server stopped.")

Server stopped.


In [44]:
!nvidia-smi

Mon Sep 21 02:06:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   78C    P0             36W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [43]:
!kill -9 16257
!kill -9 16483

In [45]:
# =============================================================================
# Start 512-token streaming server
# =============================================================================

import subprocess

SERVER_LOG = "/content/vllm_server_512.log"

server_process = subprocess.Popen(
    [
        "python",
        "-m",
        "vllm.entrypoints.openai.api_server",
        "--model",
        MODEL_NAME,
        "--dtype",
        DTYPE,
        "--max-model-len",
        str(MAX_MODEL_LEN),
        "--gpu-memory-utilization",
        str(GPU_MEMORY_UTILIZATION),
        "--max-num-batched-tokens",
        "512",
        "--enable-chunked-prefill",
        "--no-enable-prefix-caching",
        "--port",
        "8000",
    ],
    stdout=open(SERVER_LOG, "w"),
    stderr=subprocess.STDOUT,
)

print("Server PID:", server_process.pid)

Server PID: 18012


In [46]:
# =============================================================================
# Inspect server startup
# =============================================================================

import time
from pathlib import Path

time.sleep(60)

log_path = Path(SERVER_LOG)

if log_path.exists():
    print(log_path.read_text()[-5000:])

B for consumed memory (weights + non-torch), 0.42 GiB for peak activation, and 0.15 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=6696201114` (6.24 GiB) to fit into requested memory, or `--kv-cache-memory=11277357568` (10.5 GiB) to fully utilize gpu memory. Current kv cache memory in use is 6.53 GiB.
(EngineCore pid=18156) INFO 09-21 02:08:45 [jit_monitor.py:85] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=18156) INFO 09-21 02:08:45 [core.py:361] init engine (profile, create kv cache, warmup model) took 23.17 s (compilation: 1.74 s)
(EngineCore pid=18156) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
(EngineCore pid=18156) INFO 09-21 02:08:47 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(APIServe

In [47]:
# =============================================================================
# Run 512-token mixed-arrival benchmark
# =============================================================================

streaming_512_results = []

for repeat in range(5):
    long_result, short_result = await run_mixed_arrival_experiment()

    long_result["repeat"] = repeat + 1
    short_result["repeat"] = repeat + 1

    streaming_512_results.append(long_result)
    streaming_512_results.append(short_result)

streaming_512_df = pd.DataFrame(streaming_512_results)

streaming_512_df

,request,ttft_s,tpot_s,latency_s,output_tokens,stream_events,repeat
0,long,1.210944,0.020401,2.496178,64,64,1
1,short,1.159896,0.019129,1.752890,32,32,1
2,long,1.050394,0.021891,2.429542,64,64,2
3,short,1.000036,0.019258,1.597030,32,32,2
4,long,1.072046,0.022202,2.470761,64,64,3
5,short,1.019762,0.019365,1.620079,32,32,3
6,long,1.067785,0.022433,2.481068,64,64,4
7,short,1.017506,0.019415,1.619381,32,32,4
8,long,1.073295,0.022853,2.513031,64,64,5
9,short,1.022298,0.019817,1.636629,32,32,5


In [48]:
# =============================================================================
# Steady-state 512-token results
# =============================================================================

steady_512_df = streaming_512_df[
    streaming_512_df["repeat"] > 1
].copy()

steady_512_summary = (
    steady_512_df
    .groupby("request")
    .agg(
        mean_ttft_s=("ttft_s", "mean"),
        p50_ttft_s=("ttft_s", lambda x: np.percentile(x, 50)),
        p95_ttft_s=("ttft_s", lambda x: np.percentile(x, 95)),
        mean_tpot_s=("tpot_s", "mean"),
        p50_tpot_s=("tpot_s", lambda x: np.percentile(x, 50)),
        p95_tpot_s=("tpot_s", lambda x: np.percentile(x, 95)),
        mean_latency_s=("latency_s", "mean"),
        p95_latency_s=("latency_s", lambda x: np.percentile(x, 95)),
    )
    .reset_index()
)

steady_512_summary

,request,mean_ttft_s,p50_ttft_s,p95_ttft_s,mean_tpot_s,p50_tpot_s,p95_tpot_s,mean_latency_s,p95_latency_s
0,long,1.06588,1.069915,1.073108,0.022345,0.022317,0.022790,2.473601,2.508236
1,short,1.01490,1.018634,1.021918,0.019464,0.019390,0.019757,1.618280,1.634147


### summary

In [49]:
# =============================================================================
# Compare 8192 vs 512 streaming baselines
# =============================================================================

streaming_comparison = steady_default_summary.merge(
    steady_512_summary,
    on="request",
    suffixes=("_8192", "_512"),
)

streaming_comparison["mean_ttft_change_pct"] = (
    (
        streaming_comparison["mean_ttft_s_512"]
        - streaming_comparison["mean_ttft_s_8192"]
    )
    / streaming_comparison["mean_ttft_s_8192"]
    * 100
)

streaming_comparison["p95_ttft_change_pct"] = (
    (
        streaming_comparison["p95_ttft_s_512"]
        - streaming_comparison["p95_ttft_s_8192"]
    )
    / streaming_comparison["p95_ttft_s_8192"]
    * 100
)

streaming_comparison["mean_tpot_change_pct"] = (
    (
        streaming_comparison["mean_tpot_s_512"]
        - streaming_comparison["mean_tpot_s_8192"]
    )
    / streaming_comparison["mean_tpot_s_8192"]
    * 100
)

streaming_comparison["mean_latency_change_pct"] = (
    (
        streaming_comparison["mean_latency_s_512"]
        - streaming_comparison["mean_latency_s_8192"]
    )
    / streaming_comparison["mean_latency_s_8192"]
    * 100
)

streaming_comparison_table = streaming_comparison[
    [
        "request",
        "mean_ttft_s_8192",
        "mean_ttft_s_512",
        "mean_ttft_change_pct",
        "p95_ttft_s_8192",
        "p95_ttft_s_512",
        "p95_ttft_change_pct",
        "mean_tpot_s_8192",
        "mean_tpot_s_512",
        "mean_tpot_change_pct",
        "mean_latency_s_8192",
        "mean_latency_s_512",
        "mean_latency_change_pct",
    ]
].copy()

streaming_comparison_table

,request,mean_ttft_s_8192,mean_ttft_s_512,mean_ttft_change_pct,p95_ttft_s_8192,p95_ttft_s_512,p95_ttft_change_pct,mean_tpot_s_8192,mean_tpot_s_512,mean_tpot_change_pct,mean_latency_s_8192,mean_latency_s_512,mean_latency_change_pct
0,long,1.102729,1.06588,-3.341578,1.144393,1.073108,-6.229118,0.022723,0.022345,-1.666657,2.534309,2.473601,-2.395449
1,short,1.094532,1.01490,-7.275396,1.143091,1.021918,-10.600491,0.019596,0.019464,-0.673791,1.702005,1.618280,-4.919177


In [50]:
streaming_comparison_table_rounded = streaming_comparison_table.copy()

numeric_cols = streaming_comparison_table_rounded.columns.drop("request")

streaming_comparison_table_rounded[numeric_cols] = (
    streaming_comparison_table_rounded[numeric_cols].round(4)
)

streaming_comparison_table_rounded

,request,mean_ttft_s_8192,mean_ttft_s_512,mean_ttft_change_pct,p95_ttft_s_8192,p95_ttft_s_512,p95_ttft_change_pct,mean_tpot_s_8192,mean_tpot_s_512,mean_tpot_change_pct,mean_latency_s_8192,mean_latency_s_512,mean_latency_change_pct
0,long,1.1027,1.0659,-3.3416,1.1444,1.0731,-6.2291,0.0227,0.0223,-1.6667,2.5343,2.4736,-2.3954
1,short,1.0945,1.0149,-7.2754,1.1431,1.0219,-10.6005,0.0196,0.0195,-0.6738,1.7020,1.6183,-4.9192


### Streaming Comparison Summary

Reducing `max_num_batched_tokens` from 8192 to 512 improved first-token latency under the mixed-arrival workload.

The effect was strongest for the short request arriving behind the long prefill:

- mean short-request TTFT decreased by approximately 7%
- p95 short-request TTFT decreased by approximately 11%
- mean end-to-end latency decreased by approximately 5%
- TPOT remained almost unchanged

The long request also showed a smaller reduction in TTFT.

These results suggest that the smaller scheduler token budget mainly improves prefill responsiveness rather than decode efficiency.

Because TPOT changed very little, the primary benefit appears before generation begins:

```text
Long prefill
    ↓
smaller scheduling chunks
    ↓
more frequent scheduling opportunities
    ↓
lower short-request first-token latency